# Dataset Exploration

This notebook streams `candidates.jsonl` efficiently to prevent RAM bloat and generates analytical distributions.

In [ ]:
import json
import os
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Config
FILE_PATH = 'candidates.jsonl'
OUTPUT_DIR = 'data/analysis'
os.makedirs(OUTPUT_DIR, exist_ok=True)


## 1. Streaming the Data

In [ ]:
stats = {
    'total_candidates': 0,
    'open_to_work_count': 0,
}
yoe_dist = []
title_dist = Counter()
industry_dist = Counter()
country_dist = Counter()
company_size_dist = Counter()
skills_dist = Counter()
certs_dist = Counter()
notice_period_dist = Counter()
github_activity = []
recruiter_response_rate = []
interview_completion = []
saved_by_recruiters = []
last_active = Counter()

# We use streaming to handle arbitrary file sizes (e.g. 100k+ candidates)
file_size = os.path.getsize(FILE_PATH) if os.path.exists(FILE_PATH) else 0

with open(FILE_PATH, 'r', encoding='utf-8') as f:
    # Progress bar based on line count is hard for streaming, 
    # so we update a tqdm based on bytes read if possible, or just standard enumeration.
    with tqdm(total=file_size, unit='B', unit_scale=True, desc='Parsing JSONL') as pbar:
        for line in f:
            pbar.update(len(line.encode('utf-8')))
            if not line.strip():
                continue
            
            stats['total_candidates'] += 1
            try:
                cand = json.loads(line)
            except Exception:
                continue
            
            # Experience
            yoe = cand.get('years_of_experience', 0)
            yoe_dist.append(yoe)
            
            # Core Details
            if cand.get('current_title'): title_dist[cand['current_title']] += 1
            if cand.get('current_industry'): industry_dist[cand['current_industry']] += 1
            if cand.get('country'): country_dist[cand['country']] += 1
            
            # Skills
            for skill in cand.get('skills', []):
                skills_dist[skill.get('name', 'Unknown')] += 1
                
            # Certs
            for cert in cand.get('certifications', []):
                certs_dist[cert.get('name', 'Unknown')] += 1
                
            # Career for company sizes
            for job in cand.get('career_history', []):
                size = job.get('company_size', 'Unknown')
                company_size_dist[size] += 1
                
            # Redrob Signals
            signals = cand.get('redrob_signals', {})
            if signals.get('open_to_work'): stats['open_to_work_count'] += 1
            
            if 'notice_period_days' in signals:
                notice_period_dist[signals['notice_period_days']] += 1
                
            if 'github_activity_score' in signals:
                github_activity.append(signals['github_activity_score'])
                
            if 'recruiter_response_rate' in signals:
                recruiter_response_rate.append(signals['recruiter_response_rate'])
                
            if 'interview_completion_rate' in signals:
                interview_completion.append(signals['interview_completion_rate'])
                
            if 'saved_by_recruiters_30d' in signals:
                saved_by_recruiters.append(signals['saved_by_recruiters_30d'])
                
            if 'last_active_date' in signals:
                date = signals['last_active_date'][:7]  # YYYY-MM
                last_active[date] += 1


## 2. Generating Statistical Charts

In [ ]:
def plot_histogram(data, title, xlabel, bins=20):
    plt.figure(figsize=(10, 5))
    plt.hist(data, bins=bins, color='skyblue', edgecolor='black')
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel('Count')
    plt.grid(axis='y', alpha=0.75)
    plt.show()

def plot_bar_chart(counter_obj, title, top_n=10):
    labels, values = zip(*counter_obj.most_common(top_n))
    plt.figure(figsize=(10, 5))
    plt.barh(labels[::-1], values[::-1], color='lightgreen')
    plt.title(title)
    plt.xlabel('Count')
    plt.show()

print(f"Total Candidates: {stats['total_candidates']}")
if stats['total_candidates'] > 0:
    otw_pct = (stats['open_to_work_count'] / stats['total_candidates']) * 100
    print(f"Open to Work: {otw_pct:.1f}%")

plot_histogram(yoe_dist, 'Years of Experience Distribution', 'Years')
plot_bar_chart(title_dist, 'Top 10 Current Titles')
plot_bar_chart(industry_dist, 'Top 10 Industries')
plot_bar_chart(country_dist, 'Top 10 Countries')
plot_bar_chart(company_size_dist, 'Company Size Distribution')
plot_bar_chart(skills_dist, 'Top 20 Skills', top_n=20)
plot_bar_chart(certs_dist, 'Top 10 Certifications')
plot_bar_chart(notice_period_dist, 'Notice Period (Days)')
plot_histogram(github_activity, 'GitHub Activity Scores', 'Score')
plot_histogram(recruiter_response_rate, 'Recruiter Response Rates', 'Rate')
plot_histogram(interview_completion, 'Interview Completion Rates', 'Rate')
plot_histogram(saved_by_recruiters, 'Saved by Recruiters (30d)', 'Count')
plot_bar_chart(last_active, 'Last Active Month')


## 3. Export JSON Report

In [ ]:
report = {
    'total_candidates': stats['total_candidates'],
    'open_to_work_percentage': (stats['open_to_work_count'] / max(1, stats['total_candidates'])) * 100,
    'top_100_skills': dict(skills_dist.most_common(100)),
    'top_titles': dict(title_dist.most_common(20)),
    'top_industries': dict(industry_dist.most_common(20)),
    'top_countries': dict(country_dist.most_common(20)),
    'company_sizes': dict(company_size_dist),
    'top_certifications': dict(certs_dist.most_common(20)),
    'notice_periods': dict(notice_period_dist),
}

report_path = os.path.join(OUTPUT_DIR, 'report.json')
with open(report_path, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=4)
print(f"Exported JSON report to {report_path}")


## 4. Generate Markdown Summary

In [ ]:
markdown = f"""# Dataset Exploration Summary

## High-Level Metrics
- **Total Candidates Evaluated**: {stats['total_candidates']}
- **Open to Work (%)**: {report['open_to_work_percentage']:.2f}%

## Top 5 Industries
"""
for k, v in industry_dist.most_common(5):
    markdown += f"- {k} ({v} candidates)\n"
    
markdown += "\n## Top 5 Countries\n"
for k, v in country_dist.most_common(5):
    markdown += f"- {k} ({v} candidates)\n"
    
markdown += "\n## Top 10 Technical Skills\n"
for k, v in skills_dist.most_common(10):
    markdown += f"- {k} ({v} occurrences)\n"

md_path = os.path.join(OUTPUT_DIR, 'summary.md')
with open(md_path, 'w', encoding='utf-8') as f:
    f.write(markdown)

print(f"Exported Markdown summary to {md_path}")
